In [1]:
# Please edit the path according to your PC environment
# When setting up a new environment, run this program after executing "data_downloader.ipynb".

from google.colab import drive
drive.mount('/content/drive')

!ls '/content/drive/MyDrive/counterfactual_pitch_sequence_analysis/'
%cd "/content/drive/MyDrive/counterfactual_pitch_sequence_analysis/"

Mounted at /content/drive
data  dataset  program
/content/drive/MyDrive/counterfactual_pitch_sequence_analysis


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_ROOT = Path("./data/pitch_sequence")
BATTER_STATS_DATA_ROOT = Path("./data/stats/batter")

# Set the target years to 2018 and later because previous-year batter statistics are required.
TRAIN_YEARS = [2018, 2019, 2022, 2023, 2024]
TEST_YEARS = [2025]

MAX_PITCHES = 6
TARGET_COLUMN = "estimated_slg_using_speedangle"

BALL_DESCRIPTIONS = {
    "ball",
    "blocked_ball",
}

# Feature variables
PITCH_FEATURE_COLUMNS = [
    "effective_speed",
    "release_spin_rate",
    "spin_axis","pfx_x",
    "pfx_z","plate_x",
    "plate_z_norm"]

# Context information
CONTEXT_STATIC_COLUMNS = ["balls","outs_when_up",
    "inning","is_top_inning","runner_on_1b","runner_on_2b",
    "runner_on_3b","bat_score_diff","n_thruorder_pitcher"]

BATTER_STAT_COLUMNS = ["K_rate","HR_rate","ISO","AVG","OPS"]
BAD_FALLBACK_MULTIPLIER = {"K_rate": 1.10, "HR_rate": 0.90, "ISO": 0.90,"AVG": 0.90,"OPS": 0.90}

C_FEATURE_COLUMNS = (
    BATTER_STAT_COLUMNS
    + ["used_batter_stats_fallback"]
    + CONTEXT_STATIC_COLUMNS)

# Define helper functions
def add_plate_z_norm(df):
    df = df.copy()
    required_cols = ["plate_z", "sz_bot", "sz_top"]
    df["plate_z_norm"] = ((df["plate_z"] - df["sz_bot"]) / (df["sz_top"] - df["sz_bot"]))
    return df

def normalize_x_for_batter_view(df):
    df = df.copy()
    left_mask = df["stand"] == "L"
    for col in ["pfx_x", "plate_x"]:
        if col in df.columns:
            df.loc[left_mask, col] = -df.loc[left_mask, col]
    return df

def add_context_features(df):
    df = df.copy()
    df["runner_on_1b"] = df["on_1b"].notna().astype(float)
    df["runner_on_2b"] = df["on_2b"].notna().astype(float)
    df["runner_on_3b"] = df["on_3b"].notna().astype(float)
    df["is_top_inning"] = (df["inning_topbot"] == "Top").astype(float)
    return df

def preprocess_pitch_df(df):
    df = df.copy()
    df = add_plate_z_norm(df)
    df = normalize_x_for_batter_view(df)
    df = add_context_features(df)
    return df

def pad_sequence(seq, max_len=6):
    if len(seq) >= max_len:
        return seq[-max_len:]
    padded = np.zeros((max_len, seq.shape[1]), dtype=np.float32)
    padded[-len(seq):] = seq
    return padded

def get_target(last_pitch):
    desc = last_pitch["description"]
    if desc in ["swinging_strike", "swinging_strike_blocked"]:
        return 0.0
    if desc == "hit_into_play":
        xslg = last_pitch[TARGET_COLUMN]
        if pd.isna(xslg) or not np.isfinite(xslg):
            return None
        return float(xslg)
    return None

def pick_first_existing_column(df, candidates, required=True, label=""):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def make_worse_fallback_stats(base_stats):
    base_stats = np.asarray(base_stats, dtype=np.float32).copy()
    adjusted = base_stats.copy()
    for i, col in enumerate(BATTER_STAT_COLUMNS):
        adjusted[i] = adjusted[i] * BAD_FALLBACK_MULTIPLIER[col]

    adjusted[BATTER_STAT_COLUMNS.index("K_rate")] = np.clip(
        adjusted[BATTER_STAT_COLUMNS.index("K_rate")], 0.0, 1.0)

    adjusted[BATTER_STAT_COLUMNS.index("HR_rate")] = np.clip(
        adjusted[BATTER_STAT_COLUMNS.index("HR_rate")], 0.0, 1.0)

    adjusted[BATTER_STAT_COLUMNS.index("ISO")] = max(
        adjusted[BATTER_STAT_COLUMNS.index("ISO")],0.0)

    adjusted[BATTER_STAT_COLUMNS.index("AVG")] = np.clip(
        adjusted[BATTER_STAT_COLUMNS.index("AVG")], 0.0,1.0)

    adjusted[BATTER_STAT_COLUMNS.index("OPS")] = max(
        adjusted[BATTER_STAT_COLUMNS.index("OPS")], 0.0)

    return adjusted.astype(np.float32)

def load_batter_stat_info(prev_year):
    excel_path = BATTER_STATS_DATA_ROOT / f"{prev_year}.xlsx"
    df = pd.read_excel(excel_path)
    df = df.copy()

    print(f"\nloading batter stats: {excel_path}")
    print("columns:", df.columns.tolist())

    id_col = pick_first_existing_column(
        df, candidates=["player_id"],
        required=True, label="Batter ID")

    name_col = pick_first_existing_column(
        df, candidates=["Name"],
        required=False, label="Batter name")

    pa_col = pick_first_existing_column(
        df,candidates=["plateAppearances"],
        required=True,label="PA")

    so_col = pick_first_existing_column(
        df, candidates=["strikeOuts"],
        required=True,label="StrikOuts")

    hr_col = pick_first_existing_column(
        df,candidates=["homeRuns"],
        required=True, label="HomeRuns")

    avg_col = pick_first_existing_column(
        df, candidates=["avg"],
        required=True, label="AVG")

    slg_col = pick_first_existing_column(
        df,candidates=["slg"],
        required=True, label="SLG")

    ops_col = pick_first_existing_column(
        df, candidates=["ops"],
        required=True, label="OPS")

    numeric_cols = [
        id_col, pa_col,
        so_col, hr_col,
        avg_col, slg_col,
        ops_col]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    player_df = df.copy()
    player_df = player_df.dropna(
        subset=[id_col, pa_col, so_col,
            hr_col, avg_col, slg_col, ops_col])

    player_df[id_col] = player_df[id_col].astype(int)
    player_df = player_df[player_df[pa_col] > 0].copy()

    player_df["K_rate"] = player_df[so_col] / player_df[pa_col]
    player_df["HR_rate"] = player_df[hr_col] / player_df[pa_col]
    player_df["ISO"] = player_df[slg_col] - player_df[avg_col]
    player_df["AVG"] = player_df[avg_col]
    player_df["OPS"] = player_df[ops_col]

    player_df[BATTER_STAT_COLUMNS] = (
        player_df[BATTER_STAT_COLUMNS]
        .replace([np.inf, -np.inf], np.nan))

    player_df = player_df.dropna(subset=BATTER_STAT_COLUMNS).copy()

    fallback_stats_base = player_df[BATTER_STAT_COLUMNS].mean().to_numpy(dtype=np.float32)
    fallback_source = "player mean"

    fallback_stats = make_worse_fallback_stats(fallback_stats_base)
    stat_map = {int(row[id_col]): row[BATTER_STAT_COLUMNS].to_numpy(dtype=np.float32)
        for _, row in player_df.iterrows()}

    print(f"loaded batting stats: {prev_year}, "
        f"file={excel_path.name}, "
        f"players={len(stat_map)}, "
        f"features={BATTER_STAT_COLUMNS}")

    return stat_map, fallback_stats

def get_batter_stat_vector_and_flag(batter_id, stat_map, fallback_stats):
    try:
        batter_id_int = int(batter_id)
    except Exception:
        return fallback_stats.astype(np.float32), 1

    if batter_id_int in stat_map:
        return stat_map[batter_id_int].astype(np.float32), 0

    return fallback_stats.astype(np.float32), 1

def empty_dataset_result():
    X_all = np.empty((0, MAX_PITCHES, len(PITCH_FEATURE_COLUMNS)), dtype=np.float32)
    C_all = np.empty((0, len(C_FEATURE_COLUMNS)), dtype=np.float32)
    y_all = np.empty((0,), dtype=np.float32)
    meta_df = pd.DataFrame()
    return X_all, C_all, y_all, meta_df

def finalize_dataset(X_all, C_all, y_all, meta_all):
    if len(X_all) == 0:
        return empty_dataset_result()

    X_all = np.stack(X_all).astype(np.float32)
    C_all = np.stack(C_all).astype(np.float32)
    y_all = np.array(y_all, dtype=np.float32)
    meta_df = pd.DataFrame(meta_all)

    return X_all, C_all, y_all, meta_df

def make_one_sample(g, csv_path, year, stat_map, fallback_stats):
    g = g.sort_values("pitch_number")
    last_pitch = g.iloc[-1]

    if last_pitch["strikes"] != 2:
        return None

    target = get_target(last_pitch)
    if target is None:
        return None

    seq_df = g[PITCH_FEATURE_COLUMNS]
    if seq_df.isna().any().any():
        return None

    context_values = last_pitch[CONTEXT_STATIC_COLUMNS]
    if context_values.isna().any():
        return None

    seq = seq_df.to_numpy(dtype=np.float32)
    seq = pad_sequence(seq, MAX_PITCHES)

    batter_stats, used_fallback = get_batter_stat_vector_and_flag(
        batter_id=last_pitch["batter"],
        stat_map=stat_map,
        fallback_stats=fallback_stats)

    context_vector = context_values.to_numpy(dtype=np.float32)

    c_vector = np.concatenate([
        batter_stats.astype(np.float32),
        np.array([used_fallback], dtype=np.float32),
        context_vector.astype(np.float32)
    ]).astype(np.float32)

    prev_description = None
    if len(g) >= 2:
        prev_description = g.iloc[-2]["description"]

    meta = {
        "year": year,
        "prev_batting_year": year - 1,
        "source_file": csv_path.name,
        "game_pk": last_pitch["game_pk"],
        "game_date": last_pitch["game_date"] if "game_date" in last_pitch.index else None,
        "at_bat_number": last_pitch["at_bat_number"],
        "pitcher": last_pitch["pitcher"],
        "batter": last_pitch["batter"],
        "pitch_count_in_pa": len(g),
        "prev_description": prev_description,
        "last_description": last_pitch["description"],
        "used_batter_stats_fallback": used_fallback,
        "target": target}

    for col, value in zip(BATTER_STAT_COLUMNS, batter_stats):
        meta[col] = float(value)

    for col, value in zip(CONTEXT_STATIC_COLUMNS, context_vector):
        meta[f"last_{col}"] = float(value)

    return seq, c_vector, target, meta

def build_dataset_from_csv(csv_path, year, stat_map, fallback_stats):
    df = pd.read_csv(csv_path)
    df = preprocess_pitch_df(df)

    numeric_cols = (
        PITCH_FEATURE_COLUMNS
        + CONTEXT_STATIC_COLUMNS
        + [TARGET_COLUMN,
           "pitch_number","strikes","batter"])

    numeric_cols = list(dict.fromkeys(numeric_cols))
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    pa_cols = ["game_pk", "at_bat_number",
        "pitcher","batter"]

    required_cols = (
        pa_cols
        + PITCH_FEATURE_COLUMNS
        + CONTEXT_STATIC_COLUMNS
        + [TARGET_COLUMN,
           "description", "pitch_number","strikes"])

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
       print(f"skip {csv_path.name}: missing columns = {missing_cols}")
       return (
           ([], [], [], []),
           ([], [], [], [])
       )

    df = df.sort_values(pa_cols + ["pitch_number"])

    X_all = []; C_all = []; y_all = []; meta_all = []
    X_prev_ball = []; C_prev_ball = []; y_prev_ball = []; meta_prev_ball = []

    for _, g in df.groupby(pa_cols, sort=False):
        sample = make_one_sample(
            g=g,
            csv_path=csv_path,
            year=year,
            stat_map=stat_map,
            fallback_stats=fallback_stats)

        if sample is None:
            continue

        seq, c_vector, target, meta = sample

        X_all.append(seq)
        C_all.append(c_vector)
        y_all.append(target)
        meta_all.append(meta)

        if meta["prev_description"] in BALL_DESCRIPTIONS:
            X_prev_ball.append(seq)
            C_prev_ball.append(c_vector)
            y_prev_ball.append(target)
            meta_prev_ball.append(meta)

    return (
        (X_all, C_all, y_all, meta_all),
        (X_prev_ball, C_prev_ball, y_prev_ball, meta_prev_ball)
    )

def build_dataset(years):
    normal = {"X": [], "C": [], "y": [], "meta": []}
    prev_ball = {"X": [], "C": [], "y": [], "meta": []}
    batting_cache = {}

    for year in years:
        prev_year = year - 1
        if prev_year not in batting_cache:
            batting_cache[prev_year] = load_batter_stat_info(prev_year)

        stat_map, fallback_stats = batting_cache[prev_year]
        year_dir = DATA_ROOT / str(year)
        csv_files = sorted(year_dir.glob("*.csv"))

        print(f"\n{year}: {len(csv_files)} files")
        for csv_path in tqdm(csv_files):
            normal_part, prev_ball_part = build_dataset_from_csv(
                csv_path=csv_path,
                year=year,
                stat_map=stat_map,
                fallback_stats=fallback_stats)

            X, C, y, meta = normal_part
            normal["X"].extend(X)
            normal["C"].extend(C)
            normal["y"].extend(y)
            normal["meta"].extend(meta)

            X, C, y, meta = prev_ball_part
            prev_ball["X"].extend(X)
            prev_ball["C"].extend(C)
            prev_ball["y"].extend(y)
            prev_ball["meta"].extend(meta)

    normal_result = finalize_dataset(
        normal["X"], normal["C"], normal["y"], normal["meta"])

    prev_ball_result = finalize_dataset(
        prev_ball["X"], prev_ball["C"], prev_ball["y"], prev_ball["meta"])

    return {
        "normal": normal_result,
        "prev_ball": prev_ball_result,
    }

In [3]:
train_variants = build_dataset(TRAIN_YEARS)
test_variants = build_dataset(TEST_YEARS)

X_train, C_train, y_train, meta_train = train_variants["normal"]
X_test, C_test, y_test, meta_test = test_variants["normal"]

X_train_prev_ball, C_train_prev_ball, y_train_prev_ball, meta_train_prev_ball = train_variants["prev_ball"]
X_test_prev_ball, C_test_prev_ball, y_test_prev_ball, meta_test_prev_ball = test_variants["prev_ball"]

print("\n===== last-pitch dataset shapes =====")
print("X_train:", X_train.shape)
print("C_train:", C_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("C_test :", C_test.shape)
print("y_test :", y_test.shape)

print("\n===== ser-up pitch dataset shapes =====")
print("X_train_prev_ball:", X_train_prev_ball.shape)
print("C_train_prev_ball:", C_train_prev_ball.shape)
print("y_train_prev_ball:", y_train_prev_ball.shape)
print("X_test_prev_ball :", X_test_prev_ball.shape)
print("C_test_prev_ball :", C_test_prev_ball.shape)
print("y_test_prev_ball :", y_test_prev_ball.shape)

OUTPUT_DIR = Path("./dataset/last_pitch")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUTPUT_DIR / "dataset_last_pitch.npz",
    X_train=X_train,
    C_train=C_train,
    y_train=y_train,
    X_test=X_test,
    C_test=C_test,
    y_test=y_test,
    pitch_feature_columns=np.array(PITCH_FEATURE_COLUMNS),
    c_feature_columns=np.array(C_FEATURE_COLUMNS),
    batter_stat_columns=np.array(BATTER_STAT_COLUMNS),
    context_static_columns=np.array(CONTEXT_STATIC_COLUMNS),
    bad_fallback_multiplier_keys=np.array(list(BAD_FALLBACK_MULTIPLIER.keys())),
    bad_fallback_multiplier_values=np.array(list(BAD_FALLBACK_MULTIPLIER.values())),
)

meta_train.to_csv(OUTPUT_DIR / "meta_train_last_pitch.csv", index=False)
meta_test.to_csv(OUTPUT_DIR / "meta_test_last_pitch.csv", index=False)

OUTPUT_DIR_PREV_BALL = Path("./dataset/set_up")
OUTPUT_DIR_PREV_BALL.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUTPUT_DIR_PREV_BALL / "dataset_set_up.npz",
    X_train=X_train_prev_ball,
    C_train=C_train_prev_ball,
    y_train=y_train_prev_ball,
    X_test=X_test_prev_ball,
    C_test=C_test_prev_ball,
    y_test=y_test_prev_ball,
    pitch_feature_columns=np.array(PITCH_FEATURE_COLUMNS),
    c_feature_columns=np.array(C_FEATURE_COLUMNS),
    batter_stat_columns=np.array(BATTER_STAT_COLUMNS),
    context_static_columns=np.array(CONTEXT_STATIC_COLUMNS),
    bad_fallback_multiplier_keys=np.array(list(BAD_FALLBACK_MULTIPLIER.keys())),
    bad_fallback_multiplier_values=np.array(list(BAD_FALLBACK_MULTIPLIER.values())),
    ball_descriptions=np.array(sorted(BALL_DESCRIPTIONS)),
)

meta_train_prev_ball.to_csv(
    OUTPUT_DIR_PREV_BALL / "meta_train_set_up.csv",
    index=False)

meta_test_prev_ball.to_csv(
    OUTPUT_DIR_PREV_BALL / "meta_test_set_up.csv",
    index=False)

print("Saved datasets")


loading batter stats: data/stats/batter/2017.xlsx
columns: ['Season', 'Name', 'player_id', 'Team', 'team_id', 'age', 'gamesPlayed', 'groundOuts', 'airOuts', 'runs', 'doubles', 'triples', 'homeRuns', 'strikeOuts', 'baseOnBalls', 'intentionalWalks', 'hits', 'hitByPitch', 'avg', 'atBats', 'obp', 'slg', 'ops', 'caughtStealing', 'stolenBases', 'stolenBasePercentage', 'caughtStealingPercentage', 'groundIntoDoublePlay', 'numberOfPitches', 'plateAppearances', 'totalBases', 'rbi', 'leftOnBase', 'sacBunts', 'sacFlies', 'babip', 'groundOutsToAirouts', 'catchersInterference', 'atBatsPerHomeRun', 'PA']
loaded batting stats: 2017, file=2017.xlsx, players=144, features=['K_rate', 'HR_rate', 'ISO', 'AVG', 'OPS']

2018: 57 files


  0%|          | 0/57 [00:00<?, ?it/s]


loading batter stats: data/stats/batter/2018.xlsx
columns: ['Season', 'Name', 'player_id', 'Team', 'team_id', 'age', 'gamesPlayed', 'groundOuts', 'airOuts', 'runs', 'doubles', 'triples', 'homeRuns', 'strikeOuts', 'baseOnBalls', 'intentionalWalks', 'hits', 'hitByPitch', 'avg', 'atBats', 'obp', 'slg', 'ops', 'caughtStealing', 'stolenBases', 'stolenBasePercentage', 'caughtStealingPercentage', 'groundIntoDoublePlay', 'numberOfPitches', 'plateAppearances', 'totalBases', 'rbi', 'leftOnBase', 'sacBunts', 'sacFlies', 'babip', 'groundOutsToAirouts', 'catchersInterference', 'atBatsPerHomeRun', 'PA']
loaded batting stats: 2018, file=2018.xlsx, players=141, features=['K_rate', 'HR_rate', 'ISO', 'AVG', 'OPS']

2019: 61 files


  0%|          | 0/61 [00:00<?, ?it/s]


loading batter stats: data/stats/batter/2021.xlsx
columns: ['Season', 'Name', 'player_id', 'Team', 'team_id', 'age', 'gamesPlayed', 'groundOuts', 'airOuts', 'runs', 'doubles', 'triples', 'homeRuns', 'strikeOuts', 'baseOnBalls', 'intentionalWalks', 'hits', 'hitByPitch', 'avg', 'atBats', 'obp', 'slg', 'ops', 'caughtStealing', 'stolenBases', 'stolenBasePercentage', 'caughtStealingPercentage', 'groundIntoDoublePlay', 'numberOfPitches', 'plateAppearances', 'totalBases', 'rbi', 'leftOnBase', 'sacBunts', 'sacFlies', 'babip', 'groundOutsToAirouts', 'catchersInterference', 'atBatsPerHomeRun', 'PA']
loaded batting stats: 2021, file=2021.xlsx, players=132, features=['K_rate', 'HR_rate', 'ISO', 'AVG', 'OPS']

2022: 45 files


...


loading batter stats: data/stats/batter/2023.xlsx
columns: ['Season', 'Name', 'player_id', 'Team', 'team_id', 'age', 'gamesPlayed', 'groundOuts', 'airOuts', 'runs', 'doubles', 'triples', 'homeRuns', 'strikeOuts', 'baseOnBalls', 'intentionalWalks', 'hits', 'hitByPitch', 'avg', 'atBats', 'obp', 'slg', 'ops', 'caughtStealing', 'stolenBases', 'stolenBasePercentage', 'caughtStealingPercentage', 'groundIntoDoublePlay', 'numberOfPitches', 'plateAppearances', 'totalBases', 'rbi', 'leftOnBase', 'sacBunts', 'sacFlies', 'babip', 'groundOutsToAirouts', 'catchersInterference', 'atBatsPerHomeRun', 'PA']
loaded batting stats: 2023, file=2023.xlsx, players=134, features=['K_rate', 'HR_rate', 'ISO', 'AVG', 'OPS']

2024: 58 files


  0%|          | 0/58 [00:00<?, ?it/s]


loading batter stats: data/stats/batter/2024.xlsx
columns: ['Season', 'Name', 'player_id', 'Team', 'team_id', 'age', 'gamesPlayed', 'groundOuts', 'airOuts', 'runs', 'doubles', 'triples', 'homeRuns', 'strikeOuts', 'baseOnBalls', 'intentionalWalks', 'hits', 'hitByPitch', 'avg', 'atBats', 'obp', 'slg', 'ops', 'caughtStealing', 'stolenBases', 'stolenBasePercentage', 'caughtStealingPercentage', 'groundIntoDoublePlay', 'numberOfPitches', 'plateAppearances', 'totalBases', 'rbi', 'leftOnBase', 'sacBunts', 'sacFlies', 'babip', 'groundOutsToAirouts', 'catchersInterference', 'atBatsPerHomeRun', 'PA']
loaded batting stats: 2024, file=2024.xlsx, players=129, features=['K_rate', 'HR_rate', 'ISO', 'AVG', 'OPS']

2025: 52 files


  0%|          | 0/52 [00:00<?, ?it/s]


===== last-pitch dataset shapes =====
X_train: (84182, 6, 7)
C_train: (84182, 15)
y_train: (84182,)
X_test : (16487, 6, 7)
C_test : (16487, 15)
y_test : (16487,)

===== ser-up pitch dataset shapes =====
X_train_prev_ball: (28520, 6, 7)
C_train_prev_ball: (28520, 15)
y_train_prev_ball: (28520,)
X_test_prev_ball : (5680, 6, 7)
C_test_prev_ball : (5680, 15)
y_test_prev_ball : (5680,)
Saved datasets


In [4]:
from google.colab import runtime
runtime.unassign()